# 2.11 PPO：最常用的策略优化算法

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.11_ppo.ipynb)


## 为什么要学

PPO（Proximal Policy Optimization，2017）是目前**使用最广泛**的 RL 算法：

- ChatGPT 的 RLHF 用它；
- RLinf、SimpleVLA-RL、VLA-RL 等 VLA 在线强化学习框架都以 PPO（或其变体 GRPO）为核心；
- 仿真中训练机器人运动控制（如四足、人形）几乎都用它。

它解决的问题：A2C 每批数据只能用一次（on-policy），太浪费。**PPO 让同一批数据可以安全地更新很多次。**

---

## 1. 从重要性采样出发

上一课：用旧策略 $\pi_{old}$ 采的数据，估计新策略 $\pi_\theta$ 的目标，可以用重要性比值

$$
r_t(\theta)=\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)},\qquad L^{IS}(\theta)=\mathbb{E}_t\big[r_t(\theta)\hat A_t\big]
$$

在 $\theta=\theta_{old}$ 处，$\nabla L^{IS}$ 恰好等于策略梯度。但如果用这个目标**反复**优化，$\pi_\theta$ 会越走越远，$r_t$ 变得很大或很小，估计失效（还记得上一课"方差极大"的例子吗？）。

TRPO 的方案：加一个约束 $\text{KL}(\pi_{old}\|\pi_\theta)\le\delta$，但需要二阶优化，实现复杂。

## 2. PPO 的裁剪目标

PPO 用一个简单得多的办法：**把比值裁剪到 $[1-\epsilon, 1+\epsilon]$**，并取较小者：

$$
\boxed{L^{CLIP}(\theta)=\mathbb{E}_t\Big[\min\big(r_t(\theta)\hat A_t,\ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat A_t\big)\Big]}
$$

理解它（$\epsilon$ 通常取 0.2）：

- **$\hat A>0$（好动作）**：想增大 $r$（增大这个动作的概率），但 $r$ 超过 $1+\epsilon$ 后目标不再增加、梯度为 0，"见好就收"；
- **$\hat A<0$（坏动作）**：想减小 $r$，但 $r$ 低于 $1-\epsilon$ 后不再有梯度，"罚够就停"；
- 取 $\min$ 让目标成为一个**悲观的下界**：任何让目标"看起来更好"的大幅改动都被忽略。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.distributions import Categorical, Normal
from vla_ladder.envs import CartPole, Pendulum, SyncVecEnv
from vla_ladder.utils import moving_average, set_seed

r = torch.linspace(0.0, 2.0, 200); eps = 0.2
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a_, A in zip(ax, [1.0, -1.0]):
    a_.plot(r, r * A, "--", label="r*A (unclipped)")
    a_.plot(r, torch.minimum(r * A, r.clamp(1 - eps, 1 + eps) * A), lw=2, label="PPO clip objective")
    a_.axvline(1, color="gray", lw=0.5); a_.set_title(f"A = {A:+.0f}"); a_.set_xlabel("ratio r"); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()

### 练习 1：PPO 裁剪损失

In [ ]:
def ppo_policy_loss(new_logp, old_logp, adv, clip_eps=0.2):
    """返回 (loss 标量, clip_frac 被裁剪的比例)。注意：要最大化 L^CLIP，所以 loss = -L^CLIP"""
    # TODO: ratio = exp(new_logp - old_logp)；按公式计算；clip_frac = (|ratio - 1| > clip_eps) 的比例
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def ppo_policy_loss(new_logp, old_logp, adv, clip_eps=0.2):
    """返回 (loss 标量, clip_frac 被裁剪的比例)。注意：要最大化 L^CLIP，所以 loss = -L^CLIP"""
    ratio = (new_logp - old_logp).exp()
    surr1 = ratio * adv
    surr2 = ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv
    loss = -torch.minimum(surr1, surr2).mean()
    clip_frac = ((ratio - 1).abs() > clip_eps).float().mean()
    return loss, clip_frac

In [ ]:
# ✅ 自测
old = torch.zeros(4)
new = torch.log(torch.tensor([1.5, 1.5, 0.5, 0.5]))       # ratio = 1.5, 1.5, 0.5, 0.5
adv = torch.tensor([1.0, -1.0, 1.0, -1.0])
loss, cf = ppo_policy_loss(new, old, adv)
# 逐项：A>0,r=1.5 -> min(1.5,1.2)=1.2；A<0,r=1.5 -> min(-1.5,-1.2)=-1.5；A>0,r=0.5 -> min(0.5,0.8)=0.5；A<0,r=0.5 -> min(-0.5,-0.8)=-0.8
assert torch.isclose(loss, -torch.tensor([1.2, -1.5, 0.5, -0.8]).mean()) and cf == 1.0
print("通过 ✅")

> 注意第 2、3 项：**当改动方向"变差"时不裁剪**（比如坏动作的概率反而升高了，$r=1.5,\hat A<0$），梯度会把它拉回来。裁剪只阻止"过度变好"。

---

## 3. PPO 完整算法

```text
重复：
  1. 用当前策略 π_old 在 N 个并行环境中各跑 T 步，得到 N×T 条数据，记录 log π_old(a|s) 与 v(s)
  2. 用 GAE 计算优势 Â 和回报目标 R̂
  3. 重复 K 个 epoch：把数据打乱分成若干小批量，对每个小批量：
       loss = -L^CLIP + c_v · (v(s) - R̂)² - c_e · 熵
       梯度下降（带梯度裁剪）
```

工程上的关键细节（按重要性排序）：

1. **优势归一化**：每个小批量内把 $\hat A$ 标准化为均值 0、方差 1；
2. **梯度裁剪**（0.5）、合适的学习率（3e-4 左右）；
3. 监控 **approx KL** 与 **clip_frac**：KL 太大说明一次更新走太远了；
4. 观测/奖励归一化（连续控制任务很重要）。

### 练习 2：近似 KL 散度

PPO 的实现里常用一个低方差的 KL 估计（来自 John Schulman 的博客）：对 $\log r=\log\pi_\theta-\log\pi_{old}$，

$$
\widehat{\text{KL}}(\pi_{old}\|\pi_\theta)=\mathbb{E}\big[(r-1)-\log r\big]
$$

它总是非负的。实现它，并与精确 KL 比较。

In [ ]:
def approx_kl(new_logp, old_logp):
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def approx_kl(new_logp, old_logp):
    log_ratio = new_logp - old_logp
    return ((log_ratio.exp() - 1) - log_ratio).mean()

In [ ]:
# ✅ 自测
p_old = Categorical(logits=torch.tensor([0.0, 1.0, 2.0])); p_new = Categorical(logits=torch.tensor([0.1, 1.2, 1.8]))
a = p_old.sample((200000,))
est = approx_kl(p_new.log_prob(a), p_old.log_prob(a))
exact = torch.distributions.kl_divergence(p_old, p_new)
print(f"近似 KL {est:.5f}  精确 KL {exact:.5f}")
assert abs(est - exact) < 2e-3
print("通过 ✅")

### 训练 PPO（CartPole）

In [ ]:
def compute_gae(rewards, values, dones, last_values, gamma=0.99, lam=0.95):
    T = rewards.shape[0]; adv = torch.zeros_like(rewards); last = torch.zeros_like(last_values)
    for t in reversed(range(T)):
        nv = last_values if t == T - 1 else values[t + 1]; nd = 1.0 - dones[t]
        last = rewards[t] + gamma * nv * nd - values[t] + gamma * lam * nd * last
        adv[t] = last
    return adv, adv + values


class DiscreteAC(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, n_actions))
        self.critic = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def dist(self, obs):
        return Categorical(logits=self.actor(obs))

    def value(self, obs):
        return self.critic(obs).squeeze(-1)


def ppo(make_env, model, total_steps, n_envs=8, n_steps=128, epochs=4, n_minibatch=4, lr=2.5e-4, gamma=0.99,
        lam=0.95, clip_eps=0.2, vf_coef=0.5, ent_coef=0.01, max_grad_norm=0.5, reward_scale=1.0, seed=0, log_every=10):
    """通用 PPO：model 需要提供 dist(obs) 与 value(obs)。返回 (回合回报列表, 对应步数, 训练统计)"""
    set_seed(seed)
    venv = SyncVecEnv([make_env for _ in range(n_envs)])
    opt = torch.optim.Adam(model.parameters(), lr=lr, eps=1e-5)
    obs = torch.as_tensor(venv.reset(seed=seed), dtype=torch.float32)
    ep_returns, ep_steps, stats, step, it = [], [], [], 0, 0
    while step < total_steps:
        # ---------- 1. 采样 ----------
        buf = {k: [] for k in ["obs", "actions", "logp", "rewards", "dones", "values"]}
        for t in range(n_steps):
            with torch.no_grad():
                d = model.dist(obs); action = d.sample(); logp = d.log_prob(action); value = model.value(obs)
            next_obs, r, term, trunc, info = venv.step(action.numpy())
            r = torch.as_tensor(r) * reward_scale
            for i in np.nonzero(trunc & ~term)[0]:          # 截断的回合：自举最后一个观测的值
                with torch.no_grad():
                    r[i] += gamma * model.value(torch.as_tensor(info["final_obs"][i], dtype=torch.float32))
            for k, v in zip(buf, [obs, action, logp, r, torch.as_tensor(term | trunc, dtype=torch.float32), value]):
                buf[k].append(v)
            obs = torch.as_tensor(next_obs, dtype=torch.float32); step += n_envs
            ep_returns += info["episode_returns"]; ep_steps += [step] * len(info["episode_returns"])
        data = {k: torch.stack(v) for k, v in buf.items()}
        with torch.no_grad():
            adv, ret = compute_gae(data["rewards"], data["values"], data["dones"], model.value(obs), gamma, lam)
        # ---------- 2. 展平成 (T*N, ...) ----------
        flat = {k: v.reshape(n_steps * n_envs, *v.shape[2:]) for k, v in data.items()}
        flat["adv"], flat["ret"] = adv.reshape(-1), ret.reshape(-1)
        # ---------- 3. 多个 epoch 的小批量更新 ----------
        B = n_steps * n_envs; mb = B // n_minibatch
        for ep in range(epochs):
            perm = torch.randperm(B)
            for start in range(0, B, mb):
                idx = perm[start : start + mb]
                d = model.dist(flat["obs"][idx]); new_logp = d.log_prob(flat["actions"][idx])
                a = flat["adv"][idx]; a = (a - a.mean()) / (a.std() + 1e-8)          # 优势归一化
                pg_loss, clip_frac = ppo_policy_loss(new_logp, flat["logp"][idx], a, clip_eps)
                v_loss = F.mse_loss(model.value(flat["obs"][idx]), flat["ret"][idx])
                entropy = d.entropy().mean()
                loss = pg_loss + vf_coef * v_loss - ent_coef * entropy
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm); opt.step()
        with torch.no_grad():
            kl = approx_kl(model.dist(flat["obs"]).log_prob(flat["actions"]), flat["logp"]).item()
        stats.append({"kl": kl, "clip_frac": clip_frac.item(), "entropy": entropy.item()})
        it += 1
        if it % log_every == 0 and ep_returns:
            print(f"step {step:7d}  最近 20 局平均回报 {np.mean(ep_returns[-20:]):8.1f}  approx_kl {kl:.4f}  clip_frac {clip_frac:.3f}")
    return ep_returns, ep_steps, stats

model = DiscreteAC(4, 2)
rets, steps, stats = ppo(CartPole, model, total_steps=100_000, lr=7e-4, epochs=10)
plt.figure(figsize=(6, 3.5)); plt.plot(steps, rets, ".", alpha=0.2); plt.plot(steps[19:], moving_average(rets, 20))
plt.xlabel("env steps"); plt.ylabel("return"); plt.title("PPO on CartPole"); plt.show()
assert np.mean(rets[-20:]) > 300

和 A2C 相同的步数，PPO 通常学得更快、更稳。

---

## 4. 连续动作：高斯策略

机器人动作是连续的。最常用的策略：**均值由网络输出、标准差是一个可学习参数（与状态无关）的对角高斯**。

$$
\pi(\mathbf{a}|s)=\mathcal{N}\big(\boldsymbol\mu_\theta(s),\ \text{diag}(\boldsymbol\sigma^2)\big),\qquad\log\pi(\mathbf{a}|s)=\sum_{i}\log\mathcal{N}(a_i;\mu_i,\sigma_i)
$$

**多维动作的对数概率要对各维求和**（各维独立）。VLA 的动作块 $(H, D)$ 同理，要对 $H\times D$ 个维度全部求和，这在 RLinf 中计算 PPO 比值时非常关键。

### 练习 3：高斯 actor-critic

In [ ]:
class GaussianAC(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64, act_scale=1.0):
        super().__init__()
        self.mu = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, act_dim))
        self.log_std = nn.Parameter(torch.zeros(act_dim))      # 与状态无关的可学习 log σ
        self.critic = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))
        self.act_scale = act_scale

    def dist(self, obs):
        # TODO: 返回一个"对数概率会对动作维度求和"的分布。
        #  提示：torch.distributions.Independent(Normal(mu, std), 1) 会自动把最后一维求和
        raise NotImplementedError

    def value(self, obs):
        return self.critic(obs).squeeze(-1)

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class GaussianAC(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64, act_scale=1.0):
        super().__init__()
        self.mu = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, act_dim))
        self.log_std = nn.Parameter(torch.zeros(act_dim))
        self.critic = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))
        self.act_scale = act_scale

    def dist(self, obs):
        mu = self.mu(obs) * self.act_scale
        std = self.log_std.exp().expand_as(mu) * self.act_scale
        return torch.distributions.Independent(Normal(mu, std), 1)

    def value(self, obs):
        return self.critic(obs).squeeze(-1)

In [ ]:
# ✅ 自测
m = GaussianAC(3, 2)
d = m.dist(torch.randn(5, 3)); a = d.sample()
assert d.log_prob(a).shape == (5,)                                   # 对动作维度求了和
assert torch.allclose(d.log_prob(a), Normal(d.base_dist.loc, d.base_dist.scale).log_prob(a).sum(-1))
print("通过 ✅")

在 Pendulum 上训练（奖励缩小 10 倍以稳定 critic 的训练；CPU 上约 2–3 分钟）：

In [ ]:
pend_model = GaussianAC(3, 1, act_scale=2.0)
rets_p, steps_p, _ = ppo(Pendulum, pend_model, total_steps=200_000, n_envs=8, n_steps=256, epochs=10, n_minibatch=8,
                         lr=3e-4, ent_coef=0.0, reward_scale=0.1, log_every=20)
plt.figure(figsize=(6, 3.5)); plt.plot(steps_p, rets_p, ".", alpha=0.2); plt.plot(steps_p[19:], moving_average(rets_p, 20))
plt.xlabel("env steps"); plt.ylabel("return (best ~ -150)"); plt.title("PPO on Pendulum"); plt.show()
print("最后 20 局平均回报:", np.mean(rets_p[-20:]))

---

## 5. PPO 用在 VLA 上：要解决什么？

RLinf 等框架把 PPO 用在 OpenVLA / π0 上时，核心问题是"**怎么得到 $\log\pi(\mathbf{a}|s)$**"：

| VLA 类型 | 动作怎么生成 | 怎么算 log π |
|---|---|---|
| 自回归离散 token（OpenVLA） | 逐 token 采样 | 每个 token 的 log softmax 相加，和 LLM 的 RLHF 一样 |
| 流匹配（π0 / π0.5） | 从噪声积分 ODE，**是确定性的** | ODE 没有概率密度可算。RLinf 的做法是把去噪过程改造成**随机**的：每一步加噪声（flow-SDE），或者学一个噪声网络（flow-noise），这样每一步都是一个高斯，log π 就是各步高斯对数概率之和 |

另外，critic（value head）通常接在 VLA 的某个隐藏层上；动作块 $(H,D)$ 的 log π 要对所有维度求和，优势是"每个动作块"一个。这些细节在第 8 章展开。

## 小结

- PPO = 重要性比值 + 裁剪 + 多 epoch 小批量更新。
- 裁剪让目标成为悲观下界，阻止策略一次走太远。
- 关键工程细节：优势归一化、梯度裁剪、监控 approx KL 和 clip_frac。
- 连续动作用对角高斯，log π 对各维求和。
- PPO 仍是 on-policy：数据只在几个 epoch 内复用。真机上更常用 off-policy 的 SAC 类方法（下一课）。